# ENTRO-CORE: Hybrid Controller Test

## PID vs ENTRO-CORE vs Hybrid

This notebook compares the three controllers under near-critical conditions.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.getcwd()))

from entro_core.hybrid_controller import HybridController, PIDController
from entro_core.controller import create_controller
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
class SimpleSystem:
    def __init__(self, psi_0=1.8, dpsi_0=0.3):
        self.psi = psi_0
        self.dpsi = dpsi_0
        self.history = []
    
    def step(self, u, dt=0.1):
        d2psi = -0.3 * self.dpsi - 0.5 * self.psi + u
        self.dpsi += d2psi * dt
        self.psi += self.dpsi * dt
        self.history.append(self.psi)
        return self.psi
    
    def reset(self):
        self.psi = 1.8
        self.dpsi = 0.3
        self.history = []

In [ ]:
def run_simulation(controller, name):
    system = SimpleSystem()
    if hasattr(controller, 'reset'):
        controller.reset()
    
    for _ in range(200):
        result = controller.step(system.psi)
        system.step(result.u)
    
    return system.history

In [ ]:
controllers = {
    'Uncontrolled': None,
    'PID': PIDController(dt=0.1),
    'ENTRO-CORE v1': create_controller('exponential'),
    'Hybrid (threshold=1.7)': HybridController(threshold=1.7)
}

trajectories = {}
for name, controller in controllers.items():
    if controller is None:
        system = SimpleSystem()
        for _ in range(200):
            system.step(0)
        trajectories[name] = system.history
    else:
        trajectories[name] = run_simulation(controller, name)

In [ ]:
plt.figure(figsize=(12, 8))
colors = ['gray', 'blue', 'red', 'green']

for (name, traj), color in zip(trajectories.items(), colors):
    plt.plot(traj, label=name, color=color, linewidth=1.5)

plt.axhline(y=0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
plt.xlabel('Time Step (dt=0.1s)')
plt.ylabel('Ψ (Entropy State)')
plt.title('Controller Comparison: Ψ(t) Evolution')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
print("=" * 50)
print("FINAL VALUES (t=20s)")
print("=" * 50)
for name, traj in trajectories.items():
    print(f"{name:25} → Ψ_final = {traj[-1]:.3f}")
print("=" * 50)